# Recorded training experiments

This notebook starts **new experiments**. It does not resume the completed two-epoch baseline. Run initialization once, then deliberately run the training cell; expect roughly 16 minutes per training epoch plus evaluation. Do not use Run All casually. Use notebook 04 to inspect existing checkpoints without retraining. Shared logic lives in `voc_detection/`.

In [ ]:
from pathlib import Path
import sys

project_dir = next(
    path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (path / "voc_detection").is_dir() and (path / "configs").is_dir()
)
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))


## Initialize a fresh experiment
Rerunning this cell resets the model and optimizer.

In [ ]:
import torch
import mlflow
from voc_detection.project import load_config, seed_everything, make_loader
from voc_detection.model import create_model
from voc_detection.engine import train_one_epoch
from voc_detection.evaluation import evaluate_model
from voc_detection.tracking import configure_tracking, log_evaluation

config = load_config()
seed_everything(config["seed"])
device = torch.device(config["device"])
training_dataset, training_loader = make_loader(config, "train", shuffle=True)
dev_dataset, dev_loader = make_loader(config, "dev", shuffle=False)
model = create_model(config["num_classes"]).to(device)
optimizer = torch.optim.SGD(
    [parameter for parameter in model.parameters() if parameter.requires_grad],
    lr=config["learning_rate"], momentum=config["momentum"], weight_decay=config["weight_decay"],
)
configure_tracking()
print("Fresh experiment initialized; no training has run.")


## Train, evaluate, and save
This cell performs training. Each epoch gets its own checkpoint and fresh development predictions.

In [ ]:
epochs = 1
checkpoint_dir = project_dir / "mlflow_local" / "checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)
with mlflow.start_run(run_name="mobilenetv3-clean-experiment") as run:
    mlflow.log_params({**config, "epochs": epochs})
    for split in ("train", "dev"):
        mlflow.log_artifact(str(project_dir / "splits" / f"{split}.txt"), artifact_path="splits")
    for epoch in range(1, epochs + 1):
        training_result = train_one_epoch(model, training_loader, optimizer, device, epoch)
        mlflow.log_metrics(training_result, step=epoch)
        checkpoint_path = checkpoint_dir / f"{run.info.run_id}_epoch_{epoch}.pt"
        torch.save({
            "epoch": epoch, "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(), "config": config,
            "mlflow_run_id": run.info.run_id,
            "torch_rng_state": torch.get_rng_state(),
            "loader_rng_state": training_loader.generator.get_state(),
        }, checkpoint_path)
        mlflow.log_artifact(str(checkpoint_path), artifact_path="checkpoints")
        evaluation = evaluate_model(model, dev_dataset, dev_loader, device)
        log_evaluation(evaluation, epoch)
        mlflow.set_tag("completed_epochs", epoch)
        print(f"Epoch {epoch}: development mAP@0.5 = {evaluation['map50']:.2%}")
